## Description
This notebook runs the model for specified scenarios

### Imports

In [5]:
import pandas as pd
import numpy as np
import pylab as plt

import pypsa

from pathlib import Path
import gc
from itertools import product

# Import the public function from the package you completed
from energy_model import model_run

### Setting model parameters

In [6]:
# Default parameters

# model years
years = [2020, 2030, 2040, 2050]

# intra-year temporal resolution
t_res = 9 # hours

# discount rates
rt = 0.07 # technical discount rate
rs = 0.01 # social discount rate

# passenger cars conversion factor
passenger_cars_cf = 40*0.1  # km/h

# Define where the files are searched from
db_dir = "input_databases/default"

### Running functions

In [7]:
def run_scenario(
    out_dir: Path = Path("results/temporal_scenarios"),
    scenario: str = "default_tech_low_carbon",
    overwrite: bool = False,
) -> None:
    """Solve one scenario and export it; optionally skip existing results."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # model.py writes its solver log to results/solver_run.log.
    Path("results").mkdir(parents=True, exist_ok=True)

    out_file = out_dir / f"{scenario}.nc"

    if out_file.exists() and not overwrite:
        print(f"{scenario} already done, skipping")
        return

    print(f"Running {scenario}...")

    n = model_run(
        db_dir=db_dir,
        years=years,
        t_res=t_res,
        passenger_cars_cf=passenger_cars_cf,
        rt=rt,
        rs=rs,
        scenario=scenario,
    )

    n.name = scenario
    n.export_to_netcdf(out_file)

    print(f"Saved {out_file}")


def save_run(
    out_dir: Path = Path("results/temporal_scenarios"),
    scenario: str = "default_tech_low_carbon",
    overwrite: bool = False,
) -> None:
    """Run one scenario; print and continue if it fails."""
    try:
        run_scenario(
            out_dir=out_dir,
            scenario=scenario,
            overwrite=overwrite,
        )
    except Exception as e:
        print(f"{scenario} failed: {e}")
    finally:
        gc.collect()

### Setting scenario parameters and running the model

In [9]:
# Four scenarios with year-dependent multipliers.
scenarios = [
    "default_tech_low_carbon",
    "default_tech_high_carbon",
    "improved_tech_low_carbon",
    "improved_tech_high_carbon",
]

final_results = Path("results/temporal_scenarios")

for scenario in scenarios:
    save_run(
        out_dir=final_results,
        scenario=scenario,
        overwrite=False,
    )

Running default_tech_low_carbon...
2020
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 299.07it/s]


Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file /private/var/folders/fx/w8lwd2d95938qx99fyfp5kw00000gp/T/linopy-problem-5lavsmrv.lp
Reading time = 1.09 seconds
obj: 560157 rows, 260165 columns, 1319207 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2099872111296.014
Saved results/temporal_scenarios/default_tech_low_carbon.nc
Running default_tech_high_carbon...
2020
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 200.52it/s]

Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file /private/var/folders/fx/w8lwd2d95938qx99fyfp5kw00000gp/T/linopy-problem-7c04eac_.lp


Reading time = 1.12 seconds
obj: 560157 rows, 260165 columns, 1319207 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2099872111296.014
Saved results/temporal_scenarios/default_tech_high_carbon.nc
Running improved_tech_low_carbon...
2020
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 278.10it/s]


Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file /private/var/folders/fx/w8lwd2d95938qx99fyfp5kw00000gp/T/linopy-problem-d39eqi8j.lp
Reading time = 1.13 seconds
obj: 560157 rows, 260165 columns, 1319207 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2102872918752.1912
Saved results/temporal_scenarios/improved_tech_low_carbon.nc
Running improved_tech_high_carbon...
2020
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 257.52it/s]


Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file /private/var/folders/fx/w8lwd2d95938qx99fyfp5kw00000gp/T/linopy-problem-7y7il63o.lp
Reading time = 1.13 seconds
obj: 560157 rows, 260165 columns, 1319207 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2102872918752.1912
Saved results/temporal_scenarios/improved_tech_high_carbon.nc


### Setting carbon prices for carbon price sensitivity testing

In [ ]:
carbon_prices = [0, 86] + list(range(100, 325, 25))
carbon_dir = Path('results/carbon_runs_final')

for price in carbon_prices:
    save_run(out_dir=carbon_dir, co2 = price)